In [58]:
import pandas as pd
import numpy as np

In [60]:
train_df = pd.read_csv("../data/fraudTrain.csv")

In [61]:
test_df = pd.read_csv("../data/fraudTest.csv")

In [62]:
import numpy as np
import pandas as pd

BASELINE_RATE = 0.58


# ----------------------------------------------------
# 1. FUNCTION: Calculate Lift for Categorical Columns
# ----------------------------------------------------
def calculate_categorical_lift(train_df, columns, baseline=0.58):
    categorical_summaries = {}
    for col in columns:
        if col in train_df.columns:
            summary = (
                df.groupby(col)
                .agg(
                    Total_Transactions=("is_fraud", "count"),
                    Fraud_Transactions=("is_fraud", "sum"),
                )
                .reset_index()
            )
            summary["Fraud_Rate_%"] = (
                summary["Fraud_Transactions"] / summary["Total_Transactions"]
            ) * 100
            summary["Lift"] = summary["Fraud_Rate_%"] / baseline
            # Sort by highest lift
            categorical_summaries[col] = summary.sort_values(
                by="Lift", ascending=False
            )
    return categorical_summaries


# Run lift for your target categories
target_cols = ["category", "merchant", "city", "zip", "street"]
cat_lifts = calculate_categorical_lift(df, target_cols, BASELINE_RATE)

# Example: Print top 5 highest lift merchants
print("--- Top 20 Highest Risk Merchants ---")
print(cat_lifts["merchant"].head(20))


# ----------------------------------------------------
# 2. FEATURE ENGINEERING: Calculate & Bin Distance (Haversine)
# ----------------------------------------------------
def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    # Haversine formula
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c  # 6367 km is the radius of Earth
    return km * 0.621371  # Convert km to Miles


# Calculate distance in miles
train_df["distance_miles"] = haversine_distance(
    train_df["lat"], train_df["long"], train_df["merch_lat"], train_df["merch_long"]
)

# Bin the distances into meaningful intervals
distance_bins = [0, 5, 15, 30, 50, 100, np.inf]
distance_labels = ["0-5 mi", "5-15 mi", "15-30 mi", "30-50 mi", "50-100 mi", "100+ mi"]
train_df["distance_bin"] = pd.cut(
    train_df["distance_miles"], bins=distance_bins, labels=distance_labels
)


# ----------------------------------------------------
# 3. FEATURE ENGINEERING: Extract & Bin Time (Hour of Day)
# ----------------------------------------------------
# Ensure transaction time is parsed as datetime (adjust column name if needed)
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
train_df["hour"] = train_df["trans_date_trans_time"].dt.hour

# Bin the hours into logical day parts
time_bins = [0, 6, 12, 18, 24]
time_labels = [
    "Late Night (00-06)",
    "Morning (06-12)",
    "Afternoon (12-18)",
    "Evening (18-24)",
]
train_df["time_of_day"] = pd.cut(
    train_df["hour"], bins=time_bins, labels=time_labels, right=False
)


# ----------------------------------------------------
# 4. ANALYZE: Fraud Rate and Lift for Distance & Time
# ----------------------------------------------------
def get_bin_analysis(train_df, bin_column, baseline=0.58):
    summary = (
        train_df.groupby(bin_column)
        .agg(
            Total_Transactions=("is_fraud", "count"),
            Fraud_Transactions=("is_fraud", "sum"),
        )
        .reset_index()
    )
    summary["Fraud_Rate_%"] = (
        summary["Fraud_Transactions"] / summary["Total_Transactions"]
    ) * 100
    summary["Lift"] = summary["Fraud_Rate_%"] / baseline
    return summary


distance_summary = get_bin_analysis(train_df, "distance_bin", BASELINE_RATE)
time_summary = get_bin_analysis(df, "time_of_day", BASELINE_RATE)

print("\n--- Distance Bin Fraud Analysis ---")
print(distance_summary)

print("\n--- Time of Day Fraud Analysis ---")
print(time_summary)


# ----------------------------------------------------
# 5. THE ULTIMATE COMBO: Distance + Time Matrix
# ----------------------------------------------------
# Find exactly when and where the spikes happen
suspicious_matrix = (
    df.groupby(["time_of_day", "distance_bin"])
    .agg(
        Total=("is_fraud", "count"),
        Fraud=("is_fraud", "sum"),
    )
    .reset_index()
)
suspicious_matrix["Fraud_Rate_%"] = (
    suspicious_matrix["Fraud"] / suspicious_matrix["Total"]
) * 100
suspicious_matrix["Lift"] = suspicious_matrix["Fraud_Rate_%"] / BASELINE_RATE

print("\n--- Top 10 Most Suspicious Combinations (Time + Distance) ---")
print(
    suspicious_matrix.sort_values(by="Lift", ascending=False)
    .dropna()
    .head(10)
)

--- Top 20 Highest Risk Merchants ---
                                 merchant  Total_Transactions  \
337                     fraud_Kozey-Boehm                1866   
245     fraud_Herman, Treutel and Dickens                1300   
304                 fraud_Kerluke-Abshire                1838   
79                        fraud_Brown PLC                1176   
200                     fraud_Goyette Inc                1943   
616                      fraud_Terry-Huel                1996   
282                        fraud_Jast Ltd                1953   
555   fraud_Schmeler, Bashirian and Price                1968   
72                   fraud_Boyer-Reichert                1908   
372  fraud_Langworth, Boehm and Gulgowski                1969   
434        fraud_Moore, Dibbert and Koepp                1266   
31       fraud_Baumbach, Feeney and Morar                1929   
494                    fraud_Rau and Sons                2490   
193              fraud_Gleason-Macejkovic           

C:\Users\kp250\AppData\Local\Temp\ipykernel_5992\1730451465.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(bin_column)
C:\Users\kp250\AppData\Local\Temp\ipykernel_5992\1730451465.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(bin_column)
C:\Users\kp250\AppData\Local\Temp\ipykernel_5992\1730451465.py:127: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["time_of_day",

In [63]:
import pandas as pd

BASELINE_RATE = 0.58

# 1. Ensure time column is parsed and extract the hour
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
train_df["hour"] = train_df["trans_date_trans_time"].dt.hour

# 2. Group purely by hour to calculate transaction and fraud counts
hourly_analysis = (
    train_df.groupby("hour")
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

# 3. Calculate Fraud Rate and Lift
hourly_analysis["Fraud_Rate_%"] = (
    hourly_analysis["Fraud_Transactions"] / hourly_analysis["Total_Transactions"]
) * 100
hourly_analysis["Lift"] = hourly_analysis["Fraud_Rate_%"] / BASELINE_RATE

# 4. Sort by hour to see the chronological timeline, or by Lift to see peak risk
hourly_timeline = hourly_analysis.sort_values(by="hour")
hourly_risk_ranking = hourly_analysis.sort_values(by="Lift", ascending=False)

print("--- Chronological 24-Hour Fraud Timeline ---")
print(hourly_timeline.to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))

print("\n--- Hours Ranked by Highest Fraud Risk ---")
print(hourly_risk_ranking.head(24).to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))

--- Chronological 24-Hour Fraud Timeline ---
 hour  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
    0               42502                 635        1.49% 2.58
    1               42869                 658        1.53% 2.65
    2               42656                 625        1.47% 2.53
    3               42769                 609        1.42% 2.46
    4               41863                  46        0.11% 0.19
    5               42171                  60        0.14% 0.25
    6               42300                  40        0.09% 0.16
    7               42203                  56        0.13% 0.23
    8               42505                  49        0.12% 0.20
    9               42185                  47        0.11% 0.19
   10               42271                  40        0.09% 0.16
   11               42082                  42        0.10% 0.17
   12               65257                  67        0.10% 0.18
   13               65314                  80        0.12% 

In [64]:
import numpy as np
import pandas as pd

BASELINE_RATE = 0.58

# 1. Ensure your time and distance features are prepared
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
train_df["hour"] = train_df["trans_date_trans_time"].dt.hour


# Vectorized Haversine function
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 2 * np.arcsin(np.sqrt(a)) * 6367 * 0.621371  # Miles


test_df["distance_miles"] = haversine_distance(
    test_df["lat"], test_df["long"], test_df["merch_lat"], test_df["merch_long"]
)

# Bin the distances
distance_bins = [0, 5, 15, 30, 50, 100, np.inf]
distance_labels = ["0-5 mi", "5-15 mi", "15-30 mi", "30-50 mi", "50-100 mi", "100+ mi"]
train_df["distance_bin"] = pd.cut(
    train_df["distance_miles"], bins=distance_bins, labels=distance_labels
)

# 2. Complete Hour-by-Hour and Distance Aggregation
hourly_distance_matrix = (
    train_df.groupby(["hour", "distance_bin"], observed=False)
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

# 3. Calculate Fraud Rate and Lift
hourly_distance_matrix["Fraud_Rate_%"] = (
    hourly_distance_matrix["Fraud_Transactions"]
    / hourly_distance_matrix["Total_Transactions"]
) * 100
hourly_distance_matrix["Lift"] = (
    hourly_distance_matrix["Fraud_Rate_%"] / BASELINE_RATE
)

# 4. Filter out empty combinations to keep the output actionable
hourly_distance_matrix = hourly_distance_matrix[
    hourly_distance_matrix["Total_Transactions"] > 0
]

# 5. Sort by highest Lift to see the absolute most suspicious hours/distances
print("--- Top 20 Most Suspicious Hour + Distance Combinations ---")
print(
    hourly_distance_matrix.sort_values(by="Lift", ascending=False)
    .head(20)
    .to_string(index=False)
)

--- Top 20 Most Suspicious Hour + Distance Combinations ---
 hour distance_bin  Total_Transactions  Fraud_Transactions  Fraud_Rate_%     Lift
   22     30-50 mi               22773                 680      2.985992 5.148262
   23     30-50 mi               22788                 663      2.909426 5.016252
   22    50-100 mi               31536                 910      2.885591 4.975157
   23    50-100 mi               31693                 901      2.842899 4.901550
   22     15-30 mi                9363                 261      2.787568 4.806152
   23     15-30 mi                9545                 264      2.765846 4.768700
    0       0-5 mi                 230                   6      2.608696 4.497751
   23      5-15 mi                2737                  70      2.557545 4.409560
   22      5-15 mi                2927                  71      2.425692 4.182227
   22       0-5 mi                 383                   9      2.349869 4.051499
    1       0-5 mi                 212

In [70]:
import numpy as np
import pandas as pd

BASELINE_RATE = 0.58

# 1. Ensure your time and distance features are prepared
test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])
test_df["hour"] = test_df["trans_date_trans_time"].dt.hour


# Vectorized Haversine function
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 2 * np.arcsin(np.sqrt(a)) * 6367 * 0.621371  # Miles


test_df["distance_miles"] = haversine_distance(
    test_df["lat"], test_df["long"], test_df["merch_lat"], test_df["merch_long"]
)

# Bin the distances
distance_bins = [0, 5, 15, 30, 50, 100, np.inf]
distance_labels = ["0-5 mi", "5-15 mi", "15-30 mi", "30-50 mi", "50-100 mi", "100+ mi"]
test_df["distance_bin"] = pd.cut(
    test_df["distance_miles"], bins=distance_bins, labels=distance_labels
)

# 2. Complete Hour-by-Hour and Distance Aggregation
hourly_distance_matrix = (
    test_df.groupby(["hour", "distance_bin"], observed=False)
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

# 3. Calculate Fraud Rate and Lift
hourly_distance_matrix["Fraud_Rate_%"] = (
    hourly_distance_matrix["Fraud_Transactions"]
    / hourly_distance_matrix["Total_Transactions"]
) * 100
hourly_distance_matrix["Lift"] = (
    hourly_distance_matrix["Fraud_Rate_%"] / BASELINE_RATE
)

# 4. Filter out empty combinations to keep the output actionable
hourly_distance_matrix = hourly_distance_matrix[
    hourly_distance_matrix["Total_Transactions"] > 0
]

# 5. Sort by highest Lift to see the absolute most suspicious hours/distances
print("--- Top 20 Most Suspicious Hour + Distance Combinations ---")
print(
    hourly_distance_matrix.sort_values(by="Lift", ascending=False)
    .head(20)
    .to_string(index=False)
)

--- Top 20 Most Suspicious Hour + Distance Combinations ---
 hour distance_bin  Total_Transactions  Fraud_Transactions  Fraud_Rate_%     Lift
   22       0-5 mi                 148                   5      3.378378 5.824790
   22      5-15 mi                1190                  30      2.521008 4.346566
    3       0-5 mi                  89                   2      2.247191 3.874467
   23       0-5 mi                 138                   3      2.173913 3.748126
   22     15-30 mi                4077                  81      1.986755 3.425440
   23    50-100 mi               13652                 267      1.955757 3.371996
   22    50-100 mi               13309                 255      1.915997 3.303443
   23     30-50 mi                9738                 182      1.868967 3.222357
   22     30-50 mi                9664                 179      1.852235 3.193509
   23     15-30 mi                4090                  70      1.711491 2.950847
   23      5-15 mi                1180

In [72]:
import pandas as pd

BASELINE_RATE = 0.58

# 1. Parse dates and extract day features
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])

# Extract Day Name (Monday, Tuesday, etc.) and Day Number (Monday=0, Sunday=6)
train_df["day_name"] = train_df["trans_date_trans_time"].dt.day_name()
train_df["day_of_week"] = train_df["trans_date_trans_time"].dt.dayofweek

# Flag Weekends (Friday night/Saturday/Sunday or strictly Saturday/Sunday depending on standard definition)
# Here we define weekend strictly as Saturday (5) and Sunday (6)
train_df["is_weekend"] = train_df["day_of_week"].isin([5, 6]).map({True: "Weekend", False: "Weekday"})

# ----------------------------------------------------
# 2. ANALYSIS: Day-by-Day Fraud Timeline
# ----------------------------------------------------
day_analysis = (
    train_df.groupby(["day_of_week", "day_name"])
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

day_analysis["Fraud_Rate_%"] = (
    day_analysis["Fraud_Transactions"] / day_analysis["Total_Transactions"]
) * 100
day_analysis["Lift"] = day_analysis["Fraud_Rate_%"] / BASELINE_RATE

# Sort chronologically from Monday to Sunday
day_analysis = day_analysis.sort_values(by="day_of_week").drop(columns=["day_of_week"])

print("--- Day-by-Day Fraud Analysis ---")
print(day_analysis.to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))


# ----------------------------------------------------
# 3. ANALYSIS: Weekend vs. Weekday Split
# ----------------------------------------------------
weekend_analysis = (
    test_df.groupby("is_weekend")
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

weekend_analysis["Fraud_Rate_%"] = (
    weekend_analysis["Fraud_Transactions"] / weekend_analysis["Total_Transactions"]
) * 100
weekend_analysis["Lift"] = weekend_analysis["Fraud_Rate_%"] / BASELINE_RATE

print("\n--- Weekend vs. Weekday Fraud Comparison ---")
print(weekend_analysis.to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))

--- Day-by-Day Fraud Analysis ---
 day_name  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
   Monday              254282                1182        0.46% 0.80
  Tuesday              160227                 935        0.58% 1.01
Wednesday              131073                 859        0.66% 1.13
 Thursday              147285                1008        0.68% 1.18
   Friday              152272                1079        0.71% 1.22
 Saturday              200957                1227        0.61% 1.05
   Sunday              250579                1216        0.49% 0.84


KeyError: 'is_weekend'

In [74]:
import pandas as pd

BASELINE_RATE = 0.58

# 1. Parse dates and extract day features
test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])

# Extract Day Name (Monday, Tuesday, etc.) and Day Number (Monday=0, Sunday=6)
test_df["day_name"] = test_df["trans_date_trans_time"].dt.day_name()
test_df["day_of_week"] = test_df["trans_date_trans_time"].dt.dayofweek

# Flag Weekends (Friday night/Saturday/Sunday or strictly Saturday/Sunday depending on standard definition)
# Here we define weekend strictly as Saturday (5) and Sunday (6)
test_df["is_weekend"] = test_df["day_of_week"].isin([5, 6]).map({True: "Weekend", False: "Weekday"})

# ----------------------------------------------------
# 2. ANALYSIS: Day-by-Day Fraud Timeline
# ----------------------------------------------------
day_analysis = (
    test_df.groupby(["day_of_week", "day_name"])
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

day_analysis["Fraud_Rate_%"] = (
    day_analysis["Fraud_Transactions"] / day_analysis["Total_Transactions"]
) * 100
day_analysis["Lift"] = day_analysis["Fraud_Rate_%"] / BASELINE_RATE

# Sort chronologically from Monday to Sunday
day_analysis = day_analysis.sort_values(by="day_of_week").drop(columns=["day_of_week"])

print("--- Day-by-Day Fraud Analysis ---")
print(day_analysis.to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))


# ----------------------------------------------------
# 3. ANALYSIS: Weekend vs. Weekday Split
# ----------------------------------------------------
weekend_analysis = (
    test_df.groupby("is_weekend")
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

weekend_analysis["Fraud_Rate_%"] = (
    weekend_analysis["Fraud_Transactions"] / weekend_analysis["Total_Transactions"]
) * 100
weekend_analysis["Lift"] = weekend_analysis["Fraud_Rate_%"] / BASELINE_RATE

print("\n--- Weekend vs. Weekday Fraud Comparison ---")
print(weekend_analysis.to_string(index=False, formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format}))

--- Day-by-Day Fraud Analysis ---
 day_name  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
   Monday              115136                 302        0.26% 0.45
  Tuesday              110113                 331        0.30% 0.52
Wednesday               52840                 266        0.50% 0.87
 Thursday               59456                 309        0.52% 0.90
   Friday               62806                 297        0.47% 0.82
 Saturday               62270                 266        0.43% 0.74
   Sunday               93098                 374        0.40% 0.69

--- Weekend vs. Weekday Fraud Comparison ---
is_weekend  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
   Weekday              400351                1505        0.38% 0.65
   Weekend              155368                 640        0.41% 0.71


In [76]:
import pandas as pd

BASELINE_RATE = 0.58

# 1. Ensure time column is parsed and extract the month
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])

# Extract Month Number (1-12) and Month Name (January, February, etc.)
train_df["month_num"] = train_df["trans_date_trans_time"].dt.month
train_df["month_name"] = train_df["trans_date_trans_time"].dt.month_name()

# 2. Group by month to calculate transaction and fraud counts
monthly_analysis = (
    train_df.groupby(["month_num", "month_name"])
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

# 3. Calculate Fraud Rate and Lift
monthly_analysis["Fraud_Rate_%"] = (
    monthly_analysis["Fraud_Transactions"] / monthly_analysis["Total_Transactions"]
) * 100
monthly_analysis["Lift"] = monthly_analysis["Fraud_Rate_%"] / BASELINE_RATE

# 4. Sort chronologically from January to December
monthly_timeline = monthly_analysis.sort_values(by="month_num").drop(
    columns=["month_num"]
)

print("--- Month-by-Month Fraud Analysis ---")
print(
    monthly_timeline.to_string(
        index=False,
        formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format},
    )
)

--- Month-by-Month Fraud Analysis ---
month_name  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
   January              104727                 849        0.81% 1.40
  February               97657                 853        0.87% 1.51
     March              143789                 938        0.65% 1.12
     April              134970                 678        0.50% 0.87
       May              146875                 935        0.64% 1.10
      June              143811                 688        0.48% 0.82
      July               86596                 331        0.38% 0.66
    August               87359                 382        0.44% 0.75
 September               70652                 418        0.59% 1.02
   October               68758                 454        0.66% 1.14
  November               70421                 388        0.55% 0.95
  December              141060                 592        0.42% 0.72


In [78]:
import pandas as pd

BASELINE_RATE = 0.58

# 1. Ensure time column is parsed and extract the month
test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])

# Extract Month Number (1-12) and Month Name (January, February, etc.)
test_df["month_num"] = test_df["trans_date_trans_time"].dt.month
test_df["month_name"] = test_df["trans_date_trans_time"].dt.month_name()

# 2. Group by month to calculate transaction and fraud counts
monthly_analysis = (
    test_df.groupby(["month_num", "month_name"])
    .agg(
        Total_Transactions=("is_fraud", "count"),
        Fraud_Transactions=("is_fraud", "sum"),
    )
    .reset_index()
)

# 3. Calculate Fraud Rate and Lift
monthly_analysis["Fraud_Rate_%"] = (
    monthly_analysis["Fraud_Transactions"] / monthly_analysis["Total_Transactions"]
) * 100
monthly_analysis["Lift"] = monthly_analysis["Fraud_Rate_%"] / BASELINE_RATE

# 4. Sort chronologically from January to December
monthly_timeline = monthly_analysis.sort_values(by="month_num").drop(
    columns=["month_num"]
)

print("--- Month-by-Month Fraud Analysis ---")
print(
    monthly_timeline.to_string(
        index=False,
        formatters={"Fraud_Rate_%": "{:.2f}%".format, "Lift": "{:.2f}".format},
    )
)

--- Month-by-Month Fraud Analysis ---
month_name  Total_Transactions  Fraud_Transactions Fraud_Rate_% Lift
      June               30058                 133        0.44% 0.76
      July               85848                 321        0.37% 0.64
    August               88759                 415        0.47% 0.81
 September               69533                 340        0.49% 0.84
   October               69348                 384        0.55% 0.95
  November               72635                 294        0.40% 0.70
  December              139538                 258        0.18% 0.32


In [80]:
test_df["cc_num"] = test_df["cc_num"].astype(str)

In [90]:
# test_df = test_df.drop(columns=['street'])
# train_df = train_df.drop(columns=['is_weekend'])
print(train_df.columns.tolist())
print(test_df.columns.tolist())

['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'distance_miles', 'distance_bin', 'hour', 'time_of_day', 'day_name', 'day_of_week', 'is_weekend', 'month_num', 'month_name']
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'distance_miles', 'hour', 'distance_bin', 'day_name', 'day_of_week', 'is_weekend', 'month_num', 'month_name']


In [92]:
print(train_df.info())
print(test_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 32 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Unnamed: 0             1296675 non-null  int64         
 1   trans_date_trans_time  1296675 non-null  datetime64[ns]
 2   cc_num                 1296675 non-null  int64         
 3   merchant               1296675 non-null  object        
 4   category               1296675 non-null  object        
 5   amt                    1296675 non-null  float64       
 6   first                  1296675 non-null  object        
 7   last                   1296675 non-null  object        
 8   gender                 1296675 non-null  object        
 9   street                 1296675 non-null  object        
 10  city                   1296675 non-null  object        
 11  state                  1296675 non-null  object        
 12  zip                    12966

In [94]:
columns_to_keep = [
    "cc_num",
    "merchant",
    "distance_miles",
    "category",
    "amt",
    "hour",
    "trans_date_trans_time",
    "is_fraud"
]

train_df = train_df[columns_to_keep]
test_df = test_df[columns_to_keep]

print("--- Remaining columns ---")
print(train_df.columns.tolist())
print(train_df.head())
print(test_df.columns.tolist())
print(test_df.head())

--- Remaining columns ---
['cc_num', 'merchant', 'distance_miles', 'category', 'amt', 'hour', 'trans_date_trans_time', 'is_fraud']
             cc_num                            merchant  distance_miles  \
0  2703186189652095          fraud_Rippin, Kub and Mann       48.807587   
1      630423337322     fraud_Heller, Gutmann and Zieme       18.761183   
2    38859492057661                fraud_Lind-Buckridge       67.193908   
3  3534093764340240  fraud_Kutch, Hermiston and Farrell       59.411247   
4   375534208663984                 fraud_Keeling-Crist       48.161255   

        category     amt  hour trans_date_trans_time  is_fraud  
0       misc_net    4.97     0   2019-01-01 00:00:18         0  
1    grocery_pos  107.23     0   2019-01-01 00:00:44         0  
2  entertainment  220.11     0   2019-01-01 00:00:51         0  
3  gas_transport   45.00     0   2019-01-01 00:01:16         0  
4       misc_pos   41.96     0   2019-01-01 00:03:06         0  
['cc_num', 'merchant', 'dist

In [96]:
# ============================================================
# STEP 1 — Create clean chronologically sorted dataframe
# ============================================================

train_sorted = train_df.copy()

train_sorted["trans_date_trans_time"] = pd.to_datetime(
    train_sorted["trans_date_trans_time"]
)

train_sorted = train_sorted.sort_values(
    ["cc_num", "trans_date_trans_time"]
).copy()


# ============================================================
# STEP 2 — Previous transaction amount
# ============================================================

train_sorted["prev_amt"] = (
    train_sorted
    .groupby("cc_num")["amt"]
    .shift(1)
)


# ============================================================
# STEP 3 — Previous transaction amount ratio
# ============================================================

train_sorted["amt_ratio_prev"] = (
    train_sorted["amt"] /
    train_sorted["prev_amt"]
)


# ============================================================
# STEP 4 — Lifetime average BEFORE current transaction
# ============================================================

train_sorted["prev_avg_amt"] = (
    train_sorted
    .groupby("cc_num")["amt"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)


# ============================================================
# STEP 5 — Previous 5 transaction average
#          Current transaction is excluded
# ============================================================

train_sorted["prev_5_avg_amt"] = (
    train_sorted
    .groupby("cc_num")["amt"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)


# ============================================================
# STEP 6 — Number of previous transactions in last 24 hours
#          Current transaction is excluded
# ============================================================

train_sorted["tx_count_prev_24h"] = (
    train_sorted
    .groupby("cc_num", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("trans_date_trans_time")["amt"]
            .rolling(
                "24h",
                closed="left"
            )
            .count()
            .set_axis(g.index)
        ),
        include_groups=False
    )
)

# Empty 24h window = 0 previous transactions
train_sorted["tx_count_prev_24h"] = (
    train_sorted["tx_count_prev_24h"]
    .fillna(0)
    .astype("int32")
)


# ============================================================
# STEP 7 — Verify everything exists
# ============================================================

required_features = [
    "prev_amt",
    "amt_ratio_prev",
    "prev_avg_amt",
    "prev_5_avg_amt",
    "tx_count_prev_24h"
]

print("Historical features:")
print(
    [
        col for col in required_features
        if col in train_sorted.columns
    ]
)

missing = [
    col for col in required_features
    if col not in train_sorted.columns
]

if missing:
    raise KeyError(
        f"Still missing: {missing}"
    )

print("\nAll historical features created successfully.")

Historical features:
['prev_amt', 'amt_ratio_prev', 'prev_avg_amt', 'prev_5_avg_amt', 'tx_count_prev_24h']

All historical features created successfully.


In [98]:
split_date = pd.Timestamp("2020-04-01")

temporal_train_velocity = train_sorted[
    train_sorted["trans_date_trans_time"] < split_date
].copy()

temporal_val_velocity = train_sorted[
    train_sorted["trans_date_trans_time"] >= split_date
].copy()

print("Temporal train:", temporal_train_velocity.shape)
print("Temporal validation:", temporal_val_velocity.shape)

Temporal train: (1097693, 13)
Temporal validation: (198982, 13)


In [100]:
features = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt",
    "tx_count_prev_24h"
]

X_temporal_train_velocity = (
    temporal_train_velocity[features].copy()
)

y_temporal_train_velocity = (
    temporal_train_velocity["is_fraud"].copy()
)

X_temporal_val_velocity = (
    temporal_val_velocity[features].copy()
)

y_temporal_val_velocity = (
    temporal_val_velocity["is_fraud"].copy()
)

print("Train:", X_temporal_train_velocity.shape)
print("Validation:", X_temporal_val_velocity.shape)

print("\nMissing train:")
print(X_temporal_train_velocity.isna().sum())

print("\nMissing validation:")
print(X_temporal_val_velocity.isna().sum())

Train: (1097693, 6)
Validation: (198982, 6)

Missing train:
amt                    0
hour                   0
category               0
prev_avg_amt         968
prev_5_avg_amt       968
tx_count_prev_24h      0
dtype: int64

Missing validation:
amt                   0
hour                  0
category              0
prev_avg_amt         15
prev_5_avg_amt       15
tx_count_prev_24h     0
dtype: int64


In [102]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# ============================================================
# Verify input columns first
# ============================================================

print("Training columns:")
print(X_temporal_train_velocity.columns.tolist())

print("\nValidation columns:")
print(X_temporal_val_velocity.columns.tolist())


# ============================================================
# Define ONLY columns that actually exist in this experiment
# ============================================================

categorical_features_velocity = [
    "category"
]

numeric_features_velocity = [
    "amt",
    "hour",
    "prev_avg_amt",
    "prev_5_avg_amt",
    "tx_count_prev_24h"
]


# ============================================================
# Fresh preprocessor
# ============================================================

preprocessor_velocity = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features_velocity
        ),
        (
            "num",
            "passthrough",
            numeric_features_velocity
        )
    ]
)


# ============================================================
# Fresh XGBoost model
# ============================================================

xgb_velocity = XGBClassifier(
    objective="binary:logistic",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=4,
    random_state=42
)


# ============================================================
# Fresh pipeline
# ============================================================

xgb_velocity_pipeline = Pipeline([
    ("preprocessor", preprocessor_velocity),
    ("model", xgb_velocity)
])


# ============================================================
# Fit
# ============================================================

xgb_velocity_pipeline.fit(
    X_temporal_train_velocity,
    y_temporal_train_velocity
)

print("\nTraining completed successfully.")

Training columns:
['amt', 'hour', 'category', 'prev_avg_amt', 'prev_5_avg_amt', 'tx_count_prev_24h']

Validation columns:
['amt', 'hour', 'category', 'prev_avg_amt', 'prev_5_avg_amt', 'tx_count_prev_24h']

Training completed successfully.


In [103]:
X_train_transformed = (
    xgb_velocity_pipeline
    .named_steps["preprocessor"]
    .transform(X_temporal_train_velocity)
)

print(
    "Transformed training shape:",
    X_train_transformed.shape
)

Transformed training shape: (1097693, 19)


In [106]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

velocity_proba = (
    xgb_velocity_pipeline
    .predict_proba(X_temporal_val_velocity)[:, 1]
)

velocity_pred = (
    velocity_proba >= 0.5
).astype(int)

print(
    f"Temporal ROC-AUC: "
    f"{roc_auc_score(y_temporal_val_velocity, velocity_proba):.6f}"
)

print(
    f"Temporal PR-AUC:  "
    f"{average_precision_score(y_temporal_val_velocity, velocity_proba):.6f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_temporal_val_velocity,
        velocity_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_temporal_val_velocity,
        velocity_pred
    )
)

Temporal ROC-AUC: 0.999398
Temporal PR-AUC:  0.967247

Classification Report:
              precision    recall  f1-score   support

           0     0.9992    0.9998    0.9995    197819
           1     0.9557    0.8710    0.9114      1163

    accuracy                         0.9990    198982
   macro avg     0.9775    0.9354    0.9554    198982
weighted avg     0.9990    0.9990    0.9990    198982


Confusion Matrix:
[[197772     47]
 [   150   1013]]


In [112]:
features_champion = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

X_train_champion = temporal_train_velocity[features_champion].copy()
y_train_champion = temporal_train_velocity["is_fraud"].copy()

X_val_champion = temporal_val_velocity[features_champion].copy()
y_val_champion = temporal_val_velocity["is_fraud"].copy()

In [114]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

preprocessor_champion = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            ["category"]
        ),
        (
            "num",
            "passthrough",
            [
                "amt",
                "hour",
                "prev_avg_amt",
                "prev_5_avg_amt"
            ]
        )
    ]
)

xgb_champion = XGBClassifier(
    objective="binary:logistic",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=4,
    random_state=42
)

xgb_champion_pipeline = Pipeline([
    ("preprocessor", preprocessor_champion),
    ("model", xgb_champion)
])

xgb_champion_pipeline.fit(
    X_train_champion,
    y_train_champion
)

print("Champion fitted successfully.")

Champion fitted successfully.


In [116]:
champion_proba = (
    xgb_champion_pipeline
    .predict_proba(X_val_champion)[:, 1]
)

champion_pred = (
    champion_proba >= 0.5
).astype(int)

error_df = temporal_val_velocity.copy()

error_df["fraud_probability"] = champion_proba
error_df["prediction"] = champion_pred

false_negatives = error_df[
    (error_df["is_fraud"] == 1) &
    (error_df["prediction"] == 0)
].copy()

print("False negatives:", len(false_negatives))

print(
    false_negatives[
        [
            "trans_date_trans_time",
            "amt",
            "category",
            "hour",
            "prev_avg_amt",
            "prev_5_avg_amt",
            "fraud_probability"
        ]
    ]
    .sort_values("fraud_probability")
    .head(30)
)

False negatives: 150
        trans_date_trans_time     amt        category  hour  prev_avg_amt  \
1281624   2020-06-15 17:47:11   20.62   personal_care    17     57.311795   
1268010   2020-06-10 17:54:00    8.78        misc_pos    17     66.264716   
1222755   2020-05-25 22:28:41   47.97   personal_care    22     60.458984   
1214950   2020-05-23 15:40:02   48.07   personal_care    15     59.973935   
1292747   2020-06-19 22:24:46   21.53  health_fitness    22     58.983346   
1115502   2020-04-09 03:23:53    8.16        misc_pos     3     87.765332   
1295315   2020-06-21 00:07:09    9.18        misc_pos     0     85.409303   
1287524   2020-06-17 13:25:55   18.62   personal_care    13     91.643548   
1113015   2020-04-07 18:50:56   18.67       kids_pets    18     85.943004   
1186634   2020-05-10 23:48:08   18.81  health_fitness    23     94.826049   
1182900   2020-05-09 12:00:40  972.58    shopping_pos    12     57.513004   
1290374   2020-06-18 22:29:33    4.06       kids_pets  

In [118]:
false_positives = error_df[
    (error_df["is_fraud"] == 0) &
    (error_df["prediction"] == 1)
].copy()

print("False positives:", len(false_positives))

print(
    false_positives[
        [
            "trans_date_trans_time",
            "amt",
            "category",
            "hour",
            "prev_avg_amt",
            "prev_5_avg_amt",
            "fraud_probability"
        ]
    ]
    .sort_values("fraud_probability", ascending=False)
    .head(30)
)

False positives: 52
        trans_date_trans_time      amt        category  hour  prev_avg_amt  \
1161393   2020-04-29 00:53:02    17.61   gas_transport     0     65.282926   
1235533   2020-05-30 23:23:03  1072.30    shopping_net    23     91.637846   
1242650   2020-06-01 22:30:08   734.42        misc_net    22     54.246663   
1238871   2020-05-31 23:39:50   736.26        misc_net    23     55.853625   
1119076   2020-04-11 03:29:25    20.59   gas_transport     3     61.234095   
1211799   2020-05-21 22:06:43   661.00    shopping_net    22     92.610960   
1278697   2020-06-14 23:49:30  2098.72    shopping_pos    23     64.285412   
1270623   2020-06-11 23:17:21    19.07       kids_pets    23     61.068606   
1131224   2020-04-15 23:01:35   325.01   entertainment    23     60.258292   
1280857   2020-06-15 14:25:11   255.72            home    14     56.387141   
1173035   2020-05-04 18:16:05  3404.22    shopping_net    18     56.664748   
1218851   2020-05-24 22:13:57   813.44      

In [120]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

threshold_results = []

for threshold in thresholds:

    pred = (
        champion_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val_champion,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val_champion,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val_champion,
            pred,
            zero_division=0
        ),
        "predicted_fraud": pred.sum()
    })

threshold_df = pd.DataFrame(
    threshold_results
)

print(threshold_df.to_string(index=False))

 threshold  precision   recall       f1  predicted_fraud
      0.05   0.680748 0.969905 0.800000             1657
      0.10   0.784452 0.954428 0.861133             1415
      0.15   0.843678 0.946690 0.892220             1305
      0.20   0.880875 0.934652 0.906967             1234
      0.25   0.910169 0.923474 0.916773             1180
      0.30   0.924479 0.915735 0.920086             1152
      0.35   0.935226 0.906277 0.920524             1127
      0.40   0.945355 0.892519 0.918178             1098
      0.45   0.947271 0.880482 0.912656             1081
      0.50   0.951174 0.871023 0.909336             1065
      0.55   0.956897 0.858985 0.905301             1044
      0.60   0.961127 0.850387 0.902372             1029
      0.65   0.968222 0.838349 0.898618             1007
      0.70   0.970737 0.827171 0.893222              991
      0.75   0.974948 0.803095 0.880717              958
      0.80   0.981740 0.785899 0.872970              931
      0.85   0.988558 0.742906 

In [122]:
best_threshold = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("\nBest F1 threshold:")
print(best_threshold)


Best F1 threshold:
threshold             0.350000
precision             0.935226
recall                0.906277
f1                    0.920524
predicted_fraud    1127.000000
Name: 6, dtype: float64


In [124]:
# ============================================================
# 1. Prepare train and test
# ============================================================

train_history = train_df.copy()
test_history = test_df.copy()

train_history["trans_date_trans_time"] = pd.to_datetime(
    train_history["trans_date_trans_time"]
)

test_history["trans_date_trans_time"] = pd.to_datetime(
    test_history["trans_date_trans_time"]
)

train_history["cc_num"] = train_history["cc_num"].astype(str)
test_history["cc_num"] = test_history["cc_num"].astype(str)

# Dataset marker
train_history["_dataset"] = "train"
test_history["_dataset"] = "test"


# ============================================================
# 2. Combine into one chronological transaction stream
# ============================================================

combined = pd.concat(
    [
        train_history.drop(columns=["is_fraud"], errors="ignore"),
        test_history.drop(columns=["is_fraud"], errors="ignore")
    ],
    ignore_index=True
)

combined = combined.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)


# ============================================================
# 3. Previous transaction amount
# ============================================================

combined["prev_amt"] = (
    combined
    .groupby("cc_num")["amt"]
    .shift(1)
)


# ============================================================
# 4. Lifetime average BEFORE current transaction
# ============================================================

combined["prev_avg_amt"] = (
    combined
    .groupby("cc_num")["amt"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)


# ============================================================
# 5. Previous 5 transaction average BEFORE current transaction
# ============================================================

combined["prev_5_avg_amt"] = (
    combined
    .groupby("cc_num")["amt"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)

# ============================================================
# 6. Extract test transactions
# ============================================================

test_engineered = combined[
    combined["_dataset"] == "test"
].copy()


# ============================================================
# 7. Verify required features
# ============================================================

required_features = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

missing_features = [
    col
    for col in required_features
    if col not in test_engineered.columns
]

if missing_features:
    raise KeyError(
        f"Missing engineered features: {missing_features}"
    )


# ============================================================
# 8. Inspect
# ============================================================

print("Engineered test shape:", test_engineered.shape)

print("\nRequired features:")
print(
    test_engineered[required_features].head(20)
)

print("\nMissing values:")
print(
    test_engineered[required_features].isna().sum()
)

Engineered test shape: (555719, 11)

Required features:
         amt  hour        category  prev_avg_amt  prev_5_avg_amt
2065   42.32    15       kids_pets     95.637845         183.738
2066   60.11    17            home     95.612038         176.906
2067  549.72    18          travel     95.594862          85.646
2068   58.76    20     food_dining     95.814458         184.610
2069   66.89    22     food_dining     95.796549         166.176
2070   36.28     0        misc_pos     95.782585         155.560
2071  109.18     6    shopping_net     95.753853         154.352
2072   74.98    11   gas_transport     95.760333         164.166
2073   72.60    13     food_dining     95.750309          69.218
2074   80.04    20   entertainment     95.739147          71.986
2075  100.27     0     grocery_pos     95.731581          74.616
2076  176.86     1    shopping_pos     95.733767          87.414
2077   88.54     4    shopping_net     95.772826         100.950
2078   29.09     9        misc_pos

In [126]:
print(
    test_engineered[
        [
            "trans_date_trans_time",
            "cc_num",
            "amt",
            "prev_avg_amt",
            "prev_5_avg_amt"
        ]
    ].head(20)
)

     trans_date_trans_time           cc_num     amt  prev_avg_amt  \
2065   2020-06-21 15:41:32  180011453250192   42.32     95.637845   
2066   2020-06-21 17:33:11  180011453250192   60.11     95.612038   
2067   2020-06-21 18:15:55  180011453250192  549.72     95.594862   
2068   2020-06-21 20:49:21  180011453250192   58.76     95.814458   
2069   2020-06-21 22:08:56  180011453250192   66.89     95.796549   
2070   2020-06-22 00:12:18  180011453250192   36.28     95.782585   
2071   2020-06-22 06:42:46  180011453250192  109.18     95.753853   
2072   2020-06-22 11:40:08  180011453250192   74.98     95.760333   
2073   2020-06-22 13:18:39  180011453250192   72.60     95.750309   
2074   2020-06-22 20:14:56  180011453250192   80.04     95.739147   
2075   2020-06-23 00:53:07  180011453250192  100.27     95.731581   
2076   2020-06-23 01:22:23  180011453250192  176.86     95.733767   
2077   2020-06-23 04:20:10  180011453250192   88.54     95.772826   
2078   2020-06-23 09:55:54  180011

In [128]:
final_features = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

X_final_test = (
    test_engineered[final_features]
    .copy()
)

y_final_test = (
    test_df
    .sort_values("trans_date_trans_time")
    ["is_fraud"]
    .copy()
)

print("X_final_test:", X_final_test.shape)
print("y_final_test:", y_final_test.shape)

print("\nMissing values:")
print(X_final_test.isna().sum())

X_final_test: (555719, 5)
y_final_test: (555719,)

Missing values:
amt                0
hour               0
category           0
prev_avg_amt      16
prev_5_avg_amt    16
dtype: int64


In [130]:
print(
    test_engineered[
        [
            "trans_date_trans_time",
            "cc_num",
            "amt",
            "hour",
            "category",
            "prev_avg_amt",
            "prev_5_avg_amt"
        ]
    ].head()
)

     trans_date_trans_time           cc_num     amt  hour     category  \
2065   2020-06-21 15:41:32  180011453250192   42.32    15    kids_pets   
2066   2020-06-21 17:33:11  180011453250192   60.11    17         home   
2067   2020-06-21 18:15:55  180011453250192  549.72    18       travel   
2068   2020-06-21 20:49:21  180011453250192   58.76    20  food_dining   
2069   2020-06-21 22:08:56  180011453250192   66.89    22  food_dining   

      prev_avg_amt  prev_5_avg_amt  
2065     95.637845         183.738  
2066     95.612038         176.906  
2067     95.594862          85.646  
2068     95.814458         184.610  
2069     95.796549         166.176  


In [132]:
# Final feature set
final_features = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

# Features are already in cc_num + timestamp order
X_final_test = (
    test_engineered[final_features]
    .copy()
)

# Align target using the EXACT same ordering
y_final_test = (
    test_history
    .sort_values(
        ["cc_num", "trans_date_trans_time"]
    )["is_fraud"]
    .reset_index(drop=True)
)

# Reset X index too
X_final_test = X_final_test.reset_index(drop=True)

print("X_final_test:", X_final_test.shape)
print("y_final_test:", y_final_test.shape)

print("\nFraud rate:", y_final_test.mean())

print("\nMissing values:")
print(X_final_test.isna().sum())

X_final_test: (555719, 5)
y_final_test: (555719,)

Fraud rate: 0.0038598644278853163

Missing values:
amt                0
hour               0
category           0
prev_avg_amt      16
prev_5_avg_amt    16
dtype: int64


In [134]:
alignment_check = test_engineered[
    [
        "cc_num",
        "trans_date_trans_time",
        "amt",
        "hour",
        "category"
    ]
].reset_index(drop=True).head(10)

target_check = (
    test_history
    .sort_values(
        ["cc_num", "trans_date_trans_time"]
    )[
        [
            "cc_num",
            "trans_date_trans_time",
            "is_fraud"
        ]
    ]
    .reset_index(drop=True)
    .head(10)
)

print("FEATURE ORDER:")
print(alignment_check)

print("\nTARGET ORDER:")
print(target_check)

FEATURE ORDER:
            cc_num trans_date_trans_time     amt  hour       category
0  180011453250192   2020-06-21 15:41:32   42.32    15      kids_pets
1  180011453250192   2020-06-21 17:33:11   60.11    17           home
2  180011453250192   2020-06-21 18:15:55  549.72    18         travel
3  180011453250192   2020-06-21 20:49:21   58.76    20    food_dining
4  180011453250192   2020-06-21 22:08:56   66.89    22    food_dining
5  180011453250192   2020-06-22 00:12:18   36.28     0       misc_pos
6  180011453250192   2020-06-22 06:42:46  109.18     6   shopping_net
7  180011453250192   2020-06-22 11:40:08   74.98    11  gas_transport
8  180011453250192   2020-06-22 13:18:39   72.60    13    food_dining
9  180011453250192   2020-06-22 20:14:56   80.04    20  entertainment

TARGET ORDER:
            cc_num trans_date_trans_time  is_fraud
0  180011453250192   2020-06-21 15:41:32         0
1  180011453250192   2020-06-21 17:33:11         0
2  180011453250192   2020-06-21 18:15:55       

In [136]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Predict probabilities
final_test_proba = (
    xgb_champion_pipeline
    .predict_proba(X_final_test)[:, 1]
)

# Default threshold
final_test_pred = (
    final_test_proba >= 0.5
).astype(int)

# Metrics
final_roc_auc = roc_auc_score(
    y_final_test,
    final_test_proba
)

final_pr_auc = average_precision_score(
    y_final_test,
    final_test_proba
)

print(f"Final Test ROC-AUC: {final_roc_auc:.6f}")
print(f"Final Test PR-AUC:  {final_pr_auc:.6f}")

print("\nClassification Report:")
print(
    classification_report(
        y_final_test,
        final_test_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_final_test,
        final_test_pred
    )
)

Final Test ROC-AUC: 0.998569
Final Test PR-AUC:  0.933346

Classification Report:
              precision    recall  f1-score   support

           0     0.9994    0.9997    0.9995    553574
           1     0.9198    0.8336    0.8745      2145

    accuracy                         0.9991    555719
   macro avg     0.9596    0.9166    0.9370    555719
weighted avg     0.9990    0.9991    0.9991    555719


Confusion Matrix:
[[553418    156]
 [   357   1788]]


In [138]:
# Pre-selected validation threshold
final_threshold = 0.40

final_test_pred_040 = (
    final_test_proba >= final_threshold
).astype(int)

print(
    classification_report(
        y_final_test,
        final_test_pred_040,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_final_test,
        final_test_pred_040
    )
)

              precision    recall  f1-score   support

           0     0.9994    0.9996    0.9995    553574
           1     0.8926    0.8559    0.8739      2145

    accuracy                         0.9990    555719
   macro avg     0.9460    0.9278    0.9367    555719
weighted avg     0.9990    0.9990    0.9990    555719


Confusion Matrix:
[[553353    221]
 [   309   1836]]


In [140]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_040 = precision_score(
    y_final_test,
    final_test_pred_040
)

recall_040 = recall_score(
    y_final_test,
    final_test_pred_040
)

f1_040 = f1_score(
    y_final_test,
    final_test_pred_040
)

print(f"Threshold: {final_threshold}")
print(f"Precision: {precision_040:.4f}")
print(f"Recall:    {recall_040:.4f}")
print(f"F1:        {f1_040:.4f}")
print(
    "Predicted fraud:",
    final_test_pred_040.sum()
)

Threshold: 0.4
Precision: 0.8926
Recall:    0.8559
F1:        0.8739
Predicted fraud: 2057


In [144]:
final_features = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

X_train_tune = temporal_train_velocity[
    final_features
].copy()

y_train_tune = temporal_train_velocity[
    "is_fraud"
].copy()

X_val_tune = temporal_val_velocity[
    final_features
].copy()

y_val_tune = temporal_val_velocity[
    "is_fraud"
].copy()

print("Train:", X_train_tune.shape)
print("Validation:", X_val_tune.shape)

Train: (1097693, 5)
Validation: (198982, 5)


In [146]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "category"
]

numeric_features = [
    "amt",
    "hour",
    "prev_avg_amt",
    "prev_5_avg_amt"
]

tuning_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

# Fit only on temporal training data
X_train_tune_transformed = tuning_preprocessor.fit_transform(
    X_train_tune
)

# Transform validation using the already-fitted preprocessor
X_val_tune_transformed = tuning_preprocessor.transform(
    X_val_tune
)

print(
    "Transformed train:",
    X_train_tune_transformed.shape
)

print(
    "Transformed validation:",
    X_val_tune_transformed.shape
)

Transformed train: (1097693, 18)
Transformed validation: (198982, 18)


In [148]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from xgboost import XGBClassifier
import pandas as pd

depth_results = []
depth_models = {}

for depth in [4, 6, 8]:

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=depth,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=4,
        random_state=42,
        early_stopping_rounds=50
    )

    model.fit(
        X_train_tune_transformed,
        y_train_tune,
        eval_set=[
            (
                X_val_tune_transformed,
                y_val_tune
            )
        ],
        verbose=False
    )

    val_proba = model.predict_proba(
        X_val_tune_transformed
    )[:, 1]

    depth_models[depth] = model

    depth_results.append({
        "max_depth": depth,
        "best_iteration": model.best_iteration,
        "best_score": model.best_score,
        "roc_auc": roc_auc_score(
            y_val_tune,
            val_proba
        ),
        "pr_auc": average_precision_score(
            y_val_tune,
            val_proba
        )
    })

depth_results_df = pd.DataFrame(depth_results)

depth_results_df = depth_results_df.sort_values(
    "pr_auc",
    ascending=False
).reset_index(drop=True)

print(
    depth_results_df.to_string(index=False)
)

 max_depth  best_iteration  best_score  roc_auc   pr_auc
         6             408    0.968068 0.999516 0.968077
         8             255    0.967905 0.999456 0.967914
         4             999    0.967477 0.999506 0.967487


In [149]:
# ------------------------------------------------------------
# Best depth model
# ------------------------------------------------------------

best_depth = int(
    depth_results_df.loc[0, "max_depth"]
)

best_depth_model = depth_models[best_depth]

print(
    "Best max_depth:",
    best_depth
)

print(
    "Best iteration:",
    best_depth_model.best_iteration
)

print(
    "Best score:",
    best_depth_model.best_score
)

Best max_depth: 6
Best iteration: 408
Best score: 0.968067567709613


In [152]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Predictions from best depth model
tuned_val_proba = (
    best_depth_model
    .predict_proba(X_val_tune_transformed)[:, 1]
)

tuned_val_pred = (
    tuned_val_proba >= 0.5
).astype(int)

print(
    f"Temporal ROC-AUC: "
    f"{roc_auc_score(y_val_tune, tuned_val_proba):.6f}"
)

print(
    f"Temporal PR-AUC:  "
    f"{average_precision_score(y_val_tune, tuned_val_proba):.6f}"
)

print("\nClassification Report:")

print(
    classification_report(
        y_val_tune,
        tuned_val_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_val_tune,
        tuned_val_pred
    )
)

Temporal ROC-AUC: 0.999516
Temporal PR-AUC:  0.968077

Classification Report:
              precision    recall  f1-score   support

           0     0.9993    0.9997    0.9995    197819
           1     0.9522    0.8727    0.9107      1163

    accuracy                         0.9990    198982
   macro avg     0.9757    0.9362    0.9551    198982
weighted avg     0.9990    0.9990    0.9990    198982


Confusion Matrix:
[[197768     51]
 [   148   1015]]


In [154]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
import pandas as pd

min_child_results = []
min_child_models = {}

for min_child in [1, 5, 10]:

    model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=1000,
        learning_rate=0.05,

        max_depth=6,
        min_child_weight=min_child,

        subsample=0.8,
        colsample_bytree=0.8,

        eval_metric="aucpr",

        tree_method="hist",
        n_jobs=4,
        random_state=42,

        early_stopping_rounds=50
    )

    model.fit(
        X_train_tune_transformed,
        y_train_tune,

        eval_set=[
            (
                X_val_tune_transformed,
                y_val_tune
            )
        ],

        verbose=False
    )

    val_proba = model.predict_proba(
        X_val_tune_transformed
    )[:, 1]

    min_child_models[min_child] = model

    min_child_results.append({
        "min_child_weight": min_child,
        "best_iteration": model.best_iteration,
        "best_score": model.best_score,
        "roc_auc": roc_auc_score(
            y_val_tune,
            val_proba
        ),
        "pr_auc": average_precision_score(
            y_val_tune,
            val_proba
        )
    })

min_child_results_df = pd.DataFrame(
    min_child_results
)

min_child_results_df = (
    min_child_results_df
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print(
    min_child_results_df.to_string(index=False)
)

 min_child_weight  best_iteration  best_score  roc_auc   pr_auc
                5             505    0.968253 0.999457 0.968263
                1             408    0.968068 0.999516 0.968077
               10             601    0.966828 0.999451 0.966838


In [155]:
best_min_child = int(
    min_child_results_df.loc[0, "min_child_weight"]
)

best_min_child_model = min_child_models[
    best_min_child
]

print(
    "Best min_child_weight:",
    best_min_child
)

print(
    "Best iteration:",
    best_min_child_model.best_iteration
)

print(
    "Best score:",
    best_min_child_model.best_score
)

Best min_child_weight: 5
Best iteration: 505
Best score: 0.9682532643981153


In [158]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

best_val_proba = (
    best_min_child_model
    .predict_proba(
        X_val_tune_transformed
    )[:, 1]
)

best_val_pred = (
    best_val_proba >= 0.5
).astype(int)

print(
    f"Temporal ROC-AUC: "
    f"{roc_auc_score(y_val_tune, best_val_proba):.6f}"
)

print(
    f"Temporal PR-AUC:  "
    f"{average_precision_score(y_val_tune, best_val_proba):.6f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_tune,
        best_val_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_tune,
        best_val_pred
    )
)

Temporal ROC-AUC: 0.999457
Temporal PR-AUC:  0.968263

Classification Report:
              precision    recall  f1-score   support

           0     0.9992    0.9998    0.9995    197819
           1     0.9566    0.8719    0.9123      1163

    accuracy                         0.9990    198982
   macro avg     0.9779    0.9358    0.9559    198982
weighted avg     0.9990    0.9990    0.9990    198982


Confusion Matrix:
[[197773     46]
 [   149   1014]]


In [160]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
import pandas as pd

subsample_results = []
subsample_models = {}

for subsample_value in [0.7, 0.8, 1.0]:

    model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=1000,
        learning_rate=0.05,

        max_depth=6,
        min_child_weight=5,

        subsample=subsample_value,
        colsample_bytree=0.8,

        eval_metric="aucpr",

        tree_method="hist",
        n_jobs=4,
        random_state=42,

        early_stopping_rounds=50
    )

    model.fit(
        X_train_tune_transformed,
        y_train_tune,

        eval_set=[
            (
                X_val_tune_transformed,
                y_val_tune
            )
        ],

        verbose=False
    )

    val_proba = model.predict_proba(
        X_val_tune_transformed
    )[:, 1]

    subsample_models[subsample_value] = model

    subsample_results.append({
        "subsample": subsample_value,
        "best_iteration": model.best_iteration,
        "best_score": model.best_score,
        "roc_auc": roc_auc_score(
            y_val_tune,
            val_proba
        ),
        "pr_auc": average_precision_score(
            y_val_tune,
            val_proba
        )
    })

subsample_results_df = pd.DataFrame(
    subsample_results
)

subsample_results_df = (
    subsample_results_df
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print(
    subsample_results_df.to_string(index=False)
)

 subsample  best_iteration  best_score  roc_auc   pr_auc
       0.8             505    0.968253 0.999457 0.968263
       0.7             442    0.967421 0.999458 0.967430
       1.0             680    0.967175 0.999418 0.967185


In [161]:
best_subsample = float(
    subsample_results_df.loc[0, "subsample"]
)

best_subsample_model = subsample_models[
    best_subsample
]

print(
    "Best subsample:",
    best_subsample
)

print(
    "Best iteration:",
    best_subsample_model.best_iteration
)

print(
    "Best score:",
    best_subsample_model.best_score
)

Best subsample: 0.8
Best iteration: 505
Best score: 0.9682532643981153


In [162]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

best_subsample_proba = (
    best_subsample_model
    .predict_proba(
        X_val_tune_transformed
    )[:, 1]
)

best_subsample_pred = (
    best_subsample_proba >= 0.5
).astype(int)

print(
    f"Temporal ROC-AUC: "
    f"{roc_auc_score(y_val_tune, best_subsample_proba):.6f}"
)

print(
    f"Temporal PR-AUC:  "
    f"{average_precision_score(y_val_tune, best_subsample_proba):.6f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_tune,
        best_subsample_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_tune,
        best_subsample_pred
    )
)

Temporal ROC-AUC: 0.999457
Temporal PR-AUC:  0.968263

Classification Report:
              precision    recall  f1-score   support

           0     0.9992    0.9998    0.9995    197819
           1     0.9566    0.8719    0.9123      1163

    accuracy                         0.9990    198982
   macro avg     0.9779    0.9358    0.9559    198982
weighted avg     0.9990    0.9990    0.9990    198982


Confusion Matrix:
[[197773     46]
 [   149   1014]]


In [163]:
print("Model type:")
print(type(best_min_child_model))

print("\nPreprocessor type:")
print(type(tuning_preprocessor))

print("\nPreprocessor fitted:")
print(
    hasattr(
        tuning_preprocessor,
        "transformers_"
    )
)

print("\nModel best iteration:")
print(best_min_child_model.best_iteration)

print("\nModel best score:")
print(best_min_child_model.best_score)

Model type:
<class 'xgboost.sklearn.XGBClassifier'>

Preprocessor type:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>

Preprocessor fitted:
True

Model best iteration:
505

Model best score:
0.9682532643981153


In [168]:
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Models directory:",
    models_dir.resolve()
)

Models directory: C:\Users\kp250\Desktop\fraud_detection\notebooks\models


In [170]:
import joblib
import json
from pathlib import Path

models_dir = Path("models")

# ------------------------------------------------------------
# Save fitted preprocessing object
# ------------------------------------------------------------

preprocessor_path = (
    models_dir /
    "fraud_preprocessor.joblib"
)

joblib.dump(
    tuning_preprocessor,
    preprocessor_path
)


# ------------------------------------------------------------
# Save XGBoost model
# ------------------------------------------------------------

model_path = (
    models_dir /
    "fraud_xgb_model.json"
)

best_min_child_model.save_model(
    model_path
)


# ------------------------------------------------------------
# Save model metadata
# ------------------------------------------------------------

metadata = {
    "model_name": "fraud_detection_xgboost",
    "model_version": "1.0.0",

    "features": [
        "amt",
        "hour",
        "category",
        "prev_avg_amt",
        "prev_5_avg_amt"
    ],

    "categorical_features": [
        "category"
    ],

    "numeric_features": [
        "amt",
        "hour",
        "prev_avg_amt",
        "prev_5_avg_amt"
    ],

    "model_type": "XGBClassifier",

    "objective": "binary:logistic",

    "hyperparameters": {
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "learning_rate": 0.05,
        "colsample_bytree": 0.8,
        "n_estimators": 1000,
        "tree_method": "hist"
    },

    "best_iteration": int(
        best_min_child_model.best_iteration
    ),

    "validation_best_pr_auc": float(
        best_min_child_model.best_score
    ),

    "default_threshold": 0.50,

    "alternative_thresholds": {
        "higher_recall": 0.35,
        "balanced": 0.40
    }
}

metadata_path = (
    models_dir /
    "fraud_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )


print("Saved:")
print(preprocessor_path)
print(model_path)
print(metadata_path)

Saved:
models\fraud_preprocessor.joblib
models\fraud_xgb_model.json
models\fraud_model_metadata.json


In [172]:
# Load preprocessor
loaded_preprocessor = joblib.load(
    "models/fraud_preprocessor.joblib"
)

# Load XGBoost model
loaded_model = XGBClassifier()

loaded_model.load_model(
    "models/fraud_xgb_model.json"
)

print("Preprocessor loaded:")
print(type(loaded_preprocessor))

print("\nModel loaded:")
print(type(loaded_model))

print("\nModel best iteration:")
print(loaded_model.best_iteration)

Preprocessor loaded:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>

Model loaded:
<class 'xgboost.sklearn.XGBClassifier'>

Model best iteration:
505


In [174]:
import numpy as np

# Small validation sample
sample_X = X_val_tune.head(100).copy()

# Original prediction
original_transformed = tuning_preprocessor.transform(
    sample_X
)

original_proba = (
    best_min_child_model
    .predict_proba(original_transformed)[:, 1]
)

# Reloaded prediction
loaded_transformed = loaded_preprocessor.transform(
    sample_X
)

loaded_proba = (
    loaded_model
    .predict_proba(loaded_transformed)[:, 1]
)

# Compare
max_difference = np.max(
    np.abs(
        original_proba -
        loaded_proba
    )
)

print(
    "Maximum probability difference:",
    max_difference
)

print(
    "\nPredictions identical:",
    np.allclose(
        original_proba,
        loaded_proba,
        rtol=1e-6,
        atol=1e-8
    )
)

Maximum probability difference: 0.0

Predictions identical: True


In [178]:
amount_analysis = (
    temporal_train_velocity
    .assign(
        amount_bin=pd.cut(
            temporal_train_velocity["amt"],
            bins=[
                0,
                100,
                250,
                500,
                750,
                1000,
                1500,
                2500,
                5000,
                10000,
                float("inf")
            ],
            include_lowest=True
        )
    )
    .groupby("amount_bin", observed=True)
    .agg(
        transactions=("is_fraud", "size"),
        fraud_count=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
    .reset_index()
)

print(
    amount_analysis.to_string(index=False)
)

       amount_bin  transactions  fraud_count  fraud_rate
  (-0.001, 100.0]        899054         1405    0.001563
   (100.0, 250.0]        165970          203    0.001223
   (250.0, 500.0]         19451         1652    0.084931
   (500.0, 750.0]          6284          436    0.069383
  (750.0, 1000.0]          3606         1834    0.508597
 (1000.0, 1500.0]          2239          813    0.363109
 (1500.0, 2500.0]           680            0    0.000000
 (2500.0, 5000.0]           292            0    0.000000
(5000.0, 10000.0]            91            0    0.000000
   (10000.0, inf]            26            0    0.000000


In [180]:
booster = best_min_child_model.get_booster()

tree_df = booster.trees_to_dataframe()

amount_splits = tree_df[
    tree_df["Feature"].isin([
        "f0",
        "f1",
        "f2",
        "f3",
        "f4"
    ])
].copy()

print(
    amount_splits[
        [
            "Tree",
            "Node",
            "Feature",
            "Split",
            "Yes",
            "No"
        ]
    ].head(50).to_string(index=False)
)

 Tree  Node Feature    Split   Yes    No
    0    23      f2  2.00001  0-33  0-34
    1     2      f2  2.00001   1-5   1-6
    1    16      f1  2.00001  1-31  1-32
    2     6      f1  2.00001  2-13  2-14
    2    15      f3  2.00001  2-27  2-28
    2    33      f1  2.00001  2-61  2-62
    2    44      f1  2.00001  2-73  2-74
    3     4      f4  2.00001   3-9  3-10
    3    12      f4  2.00001  3-25  3-26
    3    14      f4  2.00001  3-29  3-30
    3    16      f1  2.00001  3-33  3-34
    3    21      f0  2.00001  3-39  3-40
    3    23      f2  2.00001  3-43  3-44
    3    24      f4  2.00001  3-45  3-46
    3    30      f0  2.00001  3-57  3-58
    4    16      f1  2.00001  4-33  4-34
    4    23      f2  2.00001  4-47  4-48
    5     2      f4  2.00001   5-5   5-6
    5    15      f2  2.00001  5-29  5-30
    5    16      f1  2.00001  5-31  5-32
    5    19      f2  2.00001  5-37  5-38
    5    20      f1  2.00001  5-39  5-40
    5    25      f0  2.00001  5-45  5-46
    5    30     

In [182]:
booster = best_min_child_model.get_booster()

tree_df = booster.trees_to_dataframe()

amount_splits = tree_df[
    tree_df["Feature"] == "f14"
].copy()

print(
    amount_splits[
        [
            "Tree",
            "Node",
            "Feature",
            "Split",
            "Yes",
            "No"
        ]
    ]
    .head(50)
    .to_string(index=False)
)

 Tree  Node Feature   Split  Yes   No
    0     1     f14  641.45  0-3  0-4
    0     2     f14  241.88  0-5  0-6
    0     3     f14  266.28  0-7  0-8
    0     5     f14   24.59 0-11 0-12
    0    11     f14    7.43 0-19 0-20
    0    12     f14   87.22 0-21 0-22
    0    14     f14    25.3 0-25 0-26
    0    15     f14  347.27 0-27 0-28
    0    21     f14   53.68 0-31 0-32
    0    24     f14   23.97 0-35 0-36
    0    25     f14    6.41 0-37 0-38
    0    26     f14  102.94 0-39 0-40
    0    28     f14  429.28 0-43 0-44
    1     1     f14  266.28  1-3  1-4
    1     5     f14    25.3 1-11 1-12
    1     7     f14    23.3 1-15 1-16
    1     9     f14  347.27 1-19 1-20
    1    14     f14  241.88 1-27 1-28
    1    15     f14     9.8 1-29 1-30
    1    25     f14  505.58 1-43 1-44
    1    27     f14   16.99 1-47 1-48
    1    28     f14  641.45 1-49 1-50
    1    29     f14    6.27 1-51 1-52
    1    31     f14   79.48 1-55 1-56
    1    32     f14   216.8 1-57 1-58
    1    34 

In [184]:
print("\nAmount split summary:")

print(
    amount_splits["Split"].describe()
)


Amount split summary:
count        3575.0
mean     170.693488
std      246.064486
min            1.53
25%           15.85
50%           46.93
75%          241.88
max          928.74
Name: Split, dtype: Float64


In [186]:
feature_splits = (
    best_min_child_model
    .get_booster()
    .trees_to_dataframe()
)

for feature_name in ["f16", "f17"]:
    splits = feature_splits[
        feature_splits["Feature"] == feature_name
    ]["Split"]

    print(f"\n{feature_name}")
    print(splits.describe())


f16
count       3459.0
mean      71.81292
std      18.724041
min       36.36067
25%      57.608948
50%       67.39605
75%      85.357925
max      138.45729
Name: Split, dtype: Float64

f17
count        3050.0
mean       138.2449
std      123.047828
min           6.768
25%           47.76
50%          95.484
75%         180.722
max          555.71
Name: Split, dtype: Float64


In [188]:
train_sorted["amt_ratio_recent5"] = (
    train_sorted["amt"] /
    train_sorted["prev_5_avg_amt"]
)

In [190]:
print(
    train_sorted[
        [
            "amt",
            "prev_5_avg_amt",
            "amt_ratio_recent5"
        ]
    ].head(20)
)

print("\nDistribution:")
print(
    train_sorted["amt_ratio_recent5"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

          amt  prev_5_avg_amt  amt_ratio_recent5
1017     7.27             NaN                NaN
2724    52.94           7.270           7.281981
2726    82.08          30.105           2.726457
2882    34.79          47.430           0.733502
2907    27.18          44.270           0.613960
4135     6.87          40.852           0.168168
4337     8.43          40.772           0.206760
5467   117.11          31.870           3.674616
6027    26.74          38.876           0.687828
6273   105.20          37.266           2.822949
7473     4.98          52.870           0.094193
8351   102.47          52.492           1.952107
10739  204.15          71.300           2.863254
12714   64.31          88.708           0.724963
14329  200.77          96.222           2.086529
14342   81.48         115.336           0.706458
14749   76.33         130.636           0.584295
15355   52.47         125.408           0.418394
16787    1.90          95.072           0.019985
18158    7.13       

In [192]:
split_date = pd.Timestamp("2020-04-01")

temporal_train_ratio = train_sorted[
    train_sorted["trans_date_trans_time"] < split_date
].copy()

temporal_val_ratio = train_sorted[
    train_sorted["trans_date_trans_time"] >= split_date
].copy()

In [194]:
features_ratio = [
    "amt",
    "hour",
    "category",
    "prev_avg_amt",
    "prev_5_avg_amt",
    "amt_ratio_recent5"
]

X_train_ratio = temporal_train_ratio[
    features_ratio
].copy()

y_train_ratio = temporal_train_ratio[
    "is_fraud"
].copy()

X_val_ratio = temporal_val_ratio[
    features_ratio
].copy()

y_val_ratio = temporal_val_ratio[
    "is_fraud"
].copy()

print("Train:", X_train_ratio.shape)
print("Validation:", X_val_ratio.shape)

print("\nMissing train:")
print(X_train_ratio.isna().sum())

print("\nMissing validation:")
print(X_val_ratio.isna().sum())

Train: (1097693, 6)
Validation: (198982, 6)

Missing train:
amt                    0
hour                   0
category               0
prev_avg_amt         968
prev_5_avg_amt       968
amt_ratio_recent5    968
dtype: int64

Missing validation:
amt                   0
hour                  0
category              0
prev_avg_amt         15
prev_5_avg_amt       15
amt_ratio_recent5    15
dtype: int64


In [196]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

ratio_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            ["category"]
        ),
        (
            "num",
            "passthrough",
            [
                "amt",
                "hour",
                "prev_avg_amt",
                "prev_5_avg_amt",
                "amt_ratio_recent5"
            ]
        )
    ]
)

xgb_ratio = XGBClassifier(
    objective="binary:logistic",

    n_estimators=1000,
    learning_rate=0.05,

    max_depth=6,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    eval_metric="aucpr",

    tree_method="hist",
    n_jobs=4,
    random_state=42,

    early_stopping_rounds=50
)

X_train_ratio_transformed = (
    ratio_preprocessor.fit_transform(X_train_ratio)
)

X_val_ratio_transformed = (
    ratio_preprocessor.transform(X_val_ratio)
)

xgb_ratio.fit(
    X_train_ratio_transformed,
    y_train_ratio,
    eval_set=[
        (
            X_val_ratio_transformed,
            y_val_ratio
        )
    ],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=4, num_parallel_tree=None, ...)

In [198]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

ratio_val_proba = (
    xgb_ratio
    .predict_proba(X_val_ratio_transformed)[:, 1]
)

ratio_val_pred = (
    ratio_val_proba >= 0.5
).astype(int)

print(
    "Best iteration:",
    xgb_ratio.best_iteration
)

print(
    "Best score:",
    xgb_ratio.best_score
)

print(
    f"\nTemporal ROC-AUC: "
    f"{roc_auc_score(y_val_ratio, ratio_val_proba):.6f}"
)

print(
    f"Temporal PR-AUC:  "
    f"{average_precision_score(y_val_ratio, ratio_val_proba):.6f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_ratio,
        ratio_val_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_ratio,
        ratio_val_pred
    )
)

Best iteration: 471
Best score: 0.9694893983269536

Temporal ROC-AUC: 0.999436
Temporal PR-AUC:  0.969498

Classification Report:
              precision    recall  f1-score   support

           0     0.9993    0.9998    0.9995    197819
           1     0.9585    0.8736    0.9141      1163

    accuracy                         0.9990    198982
   macro avg     0.9789    0.9367    0.9568    198982
weighted avg     0.9990    0.9990    0.9990    198982


Confusion Matrix:
[[197775     44]
 [   147   1016]]


In [206]:
import joblib
import json
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Save new fitted preprocessor
# ------------------------------------------------------------

preprocessor_path = (
    models_dir / "fraud_preprocessor.joblib"
)

joblib.dump(
    ratio_preprocessor,
    preprocessor_path
)


# ------------------------------------------------------------
# Save new XGBoost model
# ------------------------------------------------------------

model_path = (
    models_dir / "fraud_xgb_model.json"
)

xgb_ratio.save_model(
    model_path
)


# ------------------------------------------------------------
# Updated metadata
# ------------------------------------------------------------

metadata = {
    "model_name": "fraud_detection_xgboost",
    "model_version": "1.1.0",

    "features": [
        "amt",
        "hour",
        "category",
        "prev_avg_amt",
        "prev_5_avg_amt",
        "amt_ratio_recent5"
    ],

    "categorical_features": [
        "category"
    ],

    "numeric_features": [
        "amt",
        "hour",
        "prev_avg_amt",
        "prev_5_avg_amt",
        "amt_ratio_recent5"
    ],

    "model_type": "XGBClassifier",

    "objective": "binary:logistic",

    "hyperparameters": {
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "learning_rate": 0.05,
        "colsample_bytree": 0.8,
        "n_estimators": 1000,
        "tree_method": "hist",
        "early_stopping_rounds": 50
    },

    "best_iteration": int(
        xgb_ratio.best_iteration
    ),

    "validation_best_pr_auc": float(
        xgb_ratio.best_score
    ),

    "validation_pr_auc": 0.969498,

    "validation_roc_auc": 0.999436,

    "default_threshold": 0.50,

    "alternative_thresholds": {
        "higher_recall": 0.35,
        "balanced": 0.40
    }
}

metadata_path = (
    models_dir /
    "fraud_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print("Updated model artifacts:")
print(preprocessor_path)
print(model_path)
print(metadata_path)

Updated model artifacts:
models\fraud_preprocessor.joblib
models\fraud_xgb_model.json
models\fraud_model_metadata.json


In [208]:
loaded_preprocessor = joblib.load(
    "models/fraud_preprocessor.joblib"
)

loaded_model = XGBClassifier()

loaded_model.load_model(
    "models/fraud_xgb_model.json"
)

print(
    "Preprocessor fitted:",
    hasattr(
        loaded_preprocessor,
        "transformers_"
    )
)

print(
    "Model best iteration:",
    loaded_model.best_iteration
)

Preprocessor fitted: True
Model best iteration: 471


In [210]:
import numpy as np

sample_X = X_val_ratio.head(100).copy()

# Original new model
original_transformed = (
    ratio_preprocessor.transform(sample_X)
)

original_proba = (
    xgb_ratio
    .predict_proba(
        original_transformed
    )[:, 1]
)

# Reloaded saved model
loaded_transformed = (
    loaded_preprocessor.transform(sample_X)
)

loaded_proba = (
    loaded_model
    .predict_proba(
        loaded_transformed
    )[:, 1]
)

max_difference = np.max(
    np.abs(
        original_proba -
        loaded_proba
    )
)

print(
    "Maximum probability difference:",
    max_difference
)

print(
    "Predictions identical:",
    np.allclose(
        original_proba,
        loaded_proba,
        rtol=1e-6,
        atol=1e-8
    )
)

Maximum probability difference: 0.0
Predictions identical: True
